In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

In [ ]:
TASK = Path.cwd().parent.parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CONFIG = json.loads((ROOT / 'config.json').read_text())
TAG = 'ar1_y'

In [ ]:
pairs = pd.read_csv(INPUT / 'transitions.csv')
panel = pd.read_csv(INPUT / 'panel.csv', index_col='dblp_id')
q0 = panel['0'].dropna().to_numpy()
DEST = OUTPUT / TAG
DEST.mkdir(exist_ok=True)

In [ ]:
rows = []
# same OLS and residual SD convention as AR(1)-S
for age, block in pairs.groupby('age', sort=True):
    X = np.column_stack([np.ones(len(block)), block.z])
    coef = np.linalg.lstsq(X, block.z_next, rcond=None)[0]
    residual = block.z_next - X @ coef
    rows.append({'age':age, 'intercept':coef[0], 'beta':coef[1],
                 'sigma':np.sqrt(np.mean(residual**2)), 'n':len(block)})
params = pd.DataFrame(rows)

In [ ]:
params.to_csv(DEST / 'parameters.csv', index=False)
pd.DataFrame([{'scale':q0.mean(), 'n':len(q0)}]).to_csv(DEST / 'initialization.csv', index=False)
display(params)